# Structural variants in locus interaction matrices

This tutorial applies deletions, inversions, and tandem duplications to a symmetric locus interaction matrix and to paired directional motif tracks. It uses a small deterministic fixture so every cell can be run offline.

OpenMiChroM uses zero-based, half-open intervals: start is included and end is excluded. For example, [4, 8) contains loci 4, 5, 6, and 7.


## What the workflow preserves

The matrix, directional motifs, and an output-to-input index map are transformed together. Inversions reverse the selected interval and exchange forward and reverse motif directions. Deletions shorten all aligned arrays. Tandem duplications insert a second copy immediately after the selected interval.

The optional ideal-chromosome adjustment removes the mean distance-dependent background before rearrangement and adds it back afterward. This example uses that option to make the long-range policy explicit.


In [ ]:
import os
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from OpenMiChroM.StructuralVariants import (
    apply_structural_variant,
    locus_labels,
    read_locus_matrix,
    write_locus_matrix,
    write_locus_sequence,
)

TUTORIAL_MODE = os.environ.get("OPENMICHROM_TUTORIAL_MODE", "fast").lower()
TUTORIAL_FAST = TUTORIAL_MODE != "full"
np.set_printoptions(precision=3, suppress=True)
print(f"Tutorial mode: {'fast' if TUTORIAL_FAST else 'full'}")


In [ ]:
size = 12
loci = np.arange(size)
separation = np.abs(loci[:, None] - loci[None, :])

# A symmetric, finite effective-potential matrix with a distance-decay component.
interaction_matrix = (
    -0.30 * np.exp(-separation / 3.0)
    - 0.025 * np.cos((loci[:, None] + loci[None, :]) / 2.0)
)
interaction_matrix = 0.5 * (interaction_matrix + interaction_matrix.T)

forward_motifs = np.zeros(size)
reverse_motifs = np.zeros(size)
forward_motifs[[2, 7, 9]] = [0.35, 0.80, 0.55]
reverse_motifs[[3, 6, 10]] = [0.60, 0.45, 0.75]

start, end = 4, 8
assert np.allclose(interaction_matrix, interaction_matrix.T)
assert np.isfinite(interaction_matrix).all()
print(f"Input: {size} loci; variant interval [{start}, {end})")


In [ ]:
results = {
    kind: apply_structural_variant(
        interaction_matrix,
        kind,
        start,
        end,
        forward_motifs=forward_motifs,
        reverse_motifs=reverse_motifs,
        adjust_ideal_chromosome=True,
        duplicate_contacts="ideal",
    )
    for kind in ("deletion", "inversion", "duplication")
}

for kind, result in results.items():
    print(
        f"{kind:11s} shape={result.matrix.shape}, "
        f"index map={result.index_map.tolist()}"
    )


In [ ]:
expected_sizes = {"deletion": 8, "inversion": 12, "duplication": 16}
for kind, result in results.items():
    expected = expected_sizes[kind]
    assert result.matrix.shape == (expected, expected)
    assert result.index_map.shape == (expected,)
    assert result.forward_motifs.shape == (expected,)
    assert result.reverse_motifs.shape == (expected,)
    assert np.allclose(result.matrix, result.matrix.T, equal_nan=True)
    assert np.isfinite(result.matrix).all()

inversion = results["inversion"]
np.testing.assert_allclose(
    inversion.forward_motifs[start:end],
    reverse_motifs[start:end][::-1],
)
np.testing.assert_allclose(
    inversion.reverse_motifs[start:end],
    forward_motifs[start:end][::-1],
)
print("All shape, symmetry, finiteness, and motif-direction checks passed.")


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2), constrained_layout=True)
panels = [("input", interaction_matrix)] + [
    (kind, results[kind].matrix)
    for kind in ("deletion", "inversion", "duplication")
]
limit = max(abs(matrix).max() for _, matrix in panels)

for axis, (title, matrix) in zip(axes, panels):
    image = axis.imshow(matrix, cmap="coolwarm", vmin=-limit, vmax=limit)
    axis.set_title(f"{title}\n{matrix.shape[0]} loci")
    axis.set_xlabel("locus")
    axis.set_ylabel("locus")
fig.colorbar(image, ax=axes, shrink=0.75, label="effective interaction")
plt.show()


## Export a coordinated OpenMiChroM input

A locus-specific interaction table needs unique type labels, and its sequence file must use the same labels in the same order. The next cell writes both files for the inversion example, reads the matrix back, and verifies the round trip. Motif tracks and the index map are saved beside them.


In [ ]:
keep_output = os.environ.get("OPENMICHROM_KEEP_TUTORIAL_OUTPUT") == "1"
temporary = None
if keep_output:
    output_dir = Path.cwd() / "structural_variant_output"
    output_dir.mkdir(exist_ok=True)
else:
    temporary = tempfile.TemporaryDirectory(prefix="openmichrom-sv-")
    output_dir = Path(temporary.name)

labels = locus_labels(inversion.matrix.shape[0])
matrix_path = write_locus_matrix(
    output_dir / "inversion_lambdas.csv",
    inversion.matrix,
    labels=labels,
    overwrite=True,
)
sequence_path = write_locus_sequence(
    output_dir / "inversion_sequence.txt",
    labels,
    overwrite=True,
)
np.savetxt(output_dir / "inversion_forward_motifs.txt", inversion.forward_motifs)
np.savetxt(output_dir / "inversion_reverse_motifs.txt", inversion.reverse_motifs)
np.savetxt(
    output_dir / "inversion_index_map.csv",
    inversion.index_map,
    fmt="%d",
    header="input_index",
    comments="",
)

loaded_matrix, loaded_labels = read_locus_matrix(matrix_path)
np.testing.assert_allclose(loaded_matrix, inversion.matrix)
assert loaded_labels == labels

print(f"Wrote {len(list(output_dir.iterdir()))} coordinated files to {output_dir}")
print(sequence_path.read_text().splitlines()[:4])


In [ ]:
if temporary is not None:
    temporary.cleanup()
    print("Temporary tutorial output removed.")
else:
    print("Output retained because OPENMICHROM_KEEP_TUTORIAL_OUTPUT=1.")


## Interpretation and scope

These operations define array bookkeeping; they do not infer which variant is biologically correct. The ideal-chromosome curve, duplication contact policy, interval boundaries, matrix normalization, and motif probabilities must be chosen for the system being modeled. Validate transformed inputs against experimental coordinates and a scientifically approved reference before production simulations.

The reusable implementation was developed from the workflow introduced in [OpenMiChroM pull request 123](https://github.com/junioreif/OpenMiChroM/pull/123) by Miles Gantcher. Biological context for the Epha4 structural-variant system can be found in [Lupiáñez et al., Cell 2015](https://doi.org/10.1016/j.cell.2015.04.004), [Bianco et al., Nature Genetics 2018](https://doi.org/10.1038/s41588-018-0098-8), and [Andrey et al., Genome Research 2017](https://doi.org/10.1101/gr.213066.116).
